# 연습 문제 1-8, 1-9 풀이: 파라미터 초깃값, 샘플 수

1-3절 연습 문제 1-8, 1-9의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 방법으로도 구현할 수 있다.

- 본문 예제 코드는 `code_examples/ch01/` 아래 예제 노트북을 참고한다.

In [1]:
# torch 라이브러리 임포트와 시드 고정
import torch

SEED = 8
torch.manual_seed(SEED)
print(f'시드 고정: {SEED}')

시드 고정: 8


## 사전 준비

본문 1-3절의 데이터 생성, 모델, 손실 함수, 학습 루프([코드 1-28] ~ [코드 1-32])를 가져온다.
에포크마다 로그를 출력하면 실험 조합마다 수백 줄이 나오므로, 마지막 손실만 돌려주는 `train_quiet()`를 따로 만들어 사용한다.

In [2]:
SAMPLE_SIZE = 50    # 생성할 샘플의 수
LR = 0.0001         # 학습률
EPOCHS = 15         # 전체 학습 에포크
INIT = [0.96, -0.56]  # 파라미터 초깃값: 본문 [코드 1-33]과 같은 값

def generate_true_data(sample_size=SAMPLE_SIZE, true_a=4.9, true_b=0.0):
    X = torch.rand((sample_size, 1)) * 10                    # 관측시간
    Y_ideal = true_a * X ** 2 + true_b                       # 이상적인 낙하거리
    noise = torch.randn((sample_size, 1)) * Y_ideal * 0.1    # 관측 오차
    return X, Y_ideal + noise

X, Y_true = generate_true_data()
TRAIN_SIZE = int(SAMPLE_SIZE * 0.8)
X_train, Y_train = X[:TRAIN_SIZE], Y_true[:TRAIN_SIZE]
X_test, Y_test = X[TRAIN_SIZE:], Y_true[TRAIN_SIZE:]
print(f'훈련 데이터 {X_train.shape[0]}개, 평가 데이터 {X_test.shape[0]}개')

훈련 데이터 40개, 평가 데이터 10개


In [3]:
def model(parameters, X):
    return parameters[0] * X ** 2 + parameters[1]

def criterion(Y_pred, Y_true):
    return torch.mean((Y_pred - Y_true) ** 2)

def train_quiet(parameters, X, Y_true, epochs=EPOCHS, learning_rate=LR, criterion=criterion):
    """[코드 1-32]와 같은 학습 루프. 로그 대신 마지막 손실만 반환한다."""
    for _ in range(epochs):
        parameters.grad = None
        train_loss = criterion(model(parameters, X), Y_true)
        train_loss.backward()
        with torch.no_grad():
            parameters -= learning_rate * parameters.grad
    return train_loss.item()

## 연습 문제 1-8

> 파라미터의 값을 무작위로 초기화하는 대신, a의 초깃값으로 0, 5, 10을, b의 초깃값으로 -3, 0, 3을 사용하는
> 아홉 개의 서로 다른 파라미터 초깃값으로 모델을 학습한 후 결과를 확인해 보자.
> 그 결과를 바탕으로 파라미터 초깃값과 모델 학습 결과를 설명해 보자.

**초깃값 아홉 가지로 학습하기**
- a의 초깃값 0, 5, 10과 b의 초깃값 -3, 0, 3을 모두 조합
- 학습률(0.0001)과 에포크(15)는 같게 두고 초깃값만 바꾼다
- 학습 후 a, b와 훈련 손실을 표로 비교

In [4]:
print(f'{"a 초깃값":>8} {"b 초깃값":>8} | {"학습 후 a":>9} {"학습 후 b":>9} {"훈련 손실":>9}')
for a_init in [0., 5., 10.]:
    for b_init in [-3., 0., 3.]:
        parameters = torch.tensor([a_init, b_init], requires_grad=True)
        train_loss = train_quiet(parameters, X_train, Y_train)
        print(f'{a_init:8.1f} {b_init:8.1f} | {parameters[0].item():9.3f} {parameters[1].item():9.3f} {train_loss:9.2f}')

   a 초깃값    b 초깃값 |    학습 후 a    학습 후 b     훈련 손실
     0.0     -3.0 |     5.158    -2.906    385.22
     0.0      0.0 |     5.106     0.089    380.34
     0.0      3.0 |     5.055     3.083    384.46
     5.0     -3.0 |     5.171    -2.992    384.95
     5.0      0.0 |     5.119     0.002    379.83
     5.0      3.0 |     5.068     2.997    383.70
    10.0     -3.0 |     5.184    -3.078    385.70
    10.0      0.0 |     5.132    -0.084    380.33
    10.0      3.0 |     5.081     2.911    383.95


**결과 읽기**
- a는 초깃값과 무관하게 5.1 부근으로 모인다
- b는 초깃값 근처에 머문다: -3에서 시작하면 -3.1~-2.9, 3에서 시작하면 2.9~3.1
- b의 이상적인 값은 0인데도 그렇다

**왜 b만 초깃값에 붙들려 있을까? 기울기를 비교해 보자**
- 같은 학습률이면 기울기가 클수록 한 번에 많이 움직인다
- 초깃값 (a, b) = (0, 0)에서 첫 기울기의 크기를 비교

In [5]:
parameters = torch.tensor([0., 0.], requires_grad=True)
criterion(model(parameters, X_train), Y_train).backward()
print(f'a의 기울기: {parameters.grad[0].item():10.2f}')
print(f'b의 기울기: {parameters.grad[1].item():10.2f}')
print(f'기울기의 크기 비율(a / b): {abs(parameters.grad[0] / parameters.grad[1]).item():.0f}배')

a의 기울기:  -17027.11
b의 기울기:    -295.09
기울기의 크기 비율(a / b): 58배


**1-8 정리: 초깃값의 영향은 파라미터마다 다르다**
- 기울기가 큰 파라미터(a)는 초깃값을 금방 잊는다
- 기울기가 작은 파라미터(b)는 같은 학습률로는 초깃값 근처에 오래 머문다

## 연습 문제 1-9

> 1-3절의 예제에서 샘플의 수를 충분히 늘려, 학습으로 구한 두 파라미터의 값이 4.9와 0에 가까워지는지 확인해 보자.
>
> 힌트: 샘플의 수에 따라 전체 학습 에포크도 수정해야 한다.

**샘플 수를 늘려 가며 학습하기**
- (샘플 수, 에포크)를 (50, 15), (500, 200), (5000, 500)으로 바꿔 학습
- 같은 데이터 조건에서 비교하도록 매번 시드를 다시 고정
- 훈련 데이터는 샘플의 80%

In [6]:
print(f'{"샘플 수":>8} {"에포크":>6} | {"학습 후 a":>9} {"학습 후 b":>9} {"훈련 손실":>9}')
for sample_size, epochs in [(50, 15), (500, 200), (5000, 500)]:
    torch.manual_seed(SEED)
    X_all, Y_all = generate_true_data(sample_size=sample_size)
    train_size = int(sample_size * 0.8)
    parameters = torch.tensor(INIT, requires_grad=True)
    train_loss = train_quiet(parameters, X_all[:train_size], Y_all[:train_size], epochs=epochs)
    print(f'{sample_size:11d} {epochs:9d} | {parameters[0].item():12.4f} '
          f'{parameters[1].item():12.4f} {train_loss:13.2f}')

    샘플 수    에포크 |    학습 후 a    학습 후 b     훈련 손실
         50        15 |       5.1184      -0.4869        380.40
        500       200 |       4.8119      -0.4391        482.28
       5000       500 |       4.9282      -0.4829        504.69


**결과 읽기**
- a: 5.12 → 4.81 → 4.93, 이상적인 값 4.9와의 차이는 0.22 → 0.09 → 0.03으로 줄어든다
- b: -0.49 → -0.44 → -0.48, 샘플을 늘려도 0에 가까워지지 않는다(초깃값 -0.56 근처에 머문다)
- 훈련 손실은 오히려 커진다: 노이즈가 낙하거리에 비례해 관측시간이 큰 샘플의 오차가 크기 때문
- 손실의 크기와 파라미터의 정확도는 별개

**직접 해 보기: 샘플이 5,000개일 때 에포크만 바꾸면?**
- 힌트는 샘플 수에 따라 에포크도 수정하라고 한다. 정말 에포크가 문제일까?
- 에포크를 15, 500, 5,000, 50,000으로 바꿨을 때 a와 b가 어떻게 달라질지 먼저 예측해 보자

In [7]:
torch.manual_seed(SEED)
X_all, Y_all = generate_true_data(sample_size=5000)
print(f'{"에포크":>8} | {"학습 후 a":>9} {"학습 후 b":>9}')
for epochs in [15, 500, 5000, 50000]:
    parameters = torch.tensor(INIT, requires_grad=True)
    train_quiet(parameters, X_all[:4000], Y_all[:4000], epochs=epochs)
    print(f'{epochs:11d} | {parameters[0].item():12.4f} {parameters[1].item():12.4f}')

     에포크 |    학습 후 a    학습 후 b
         15 |       4.9268      -0.4940


        500 |       4.9282      -0.4829
       5000 |       4.9268      -0.3997
      50000 |       4.9241      -0.2332


**확인**
- a는 15에포크에서 이미 4.93 부근에 도달한다: 에포크를 늘려도 달라지지 않는다
- b는 에포크를 3,000배로 늘려도(50,000) -0.23까지만 움직인다
- 샘플 수를 늘려도 에포크를 늘려도 b가 0에 가까워지지 않는 이유는 b의 기울기가 작기 때문(1-8)